# Phase 3 — The 2026 approach: CLIP zero-shot (actually run)

> **This is not a design note.** The modern approach was *run* on the same 1,050
> Flipkart products and measured. Full pipeline: `src/run_clip_zeroshot.py` →
> `results_clip_zeroshot.json`.

In 2023 the hard part was *fusing* two unrelated feature spaces (Word2Vec vs
VGG16) — hence the early-vs-late fusion study in notebook 02. **CLIP removes the
problem by construction:** it is trained so an image and its text land at the
*same place* in one vector space. Classification, retrieval and even zero-shot
labelling then become cosine-similarity lookups.

## 1 · Zero-shot classification — no training at all

With CLIP you classify into the 7 categories **without a single labelled
example**: embed the category *names* as text, embed each product (image and/or
text), and assign the nearest category by cosine similarity. In 2023 this needed
K-Means + manual Hungarian alignment; here it is a dot product.

In [ ]:
from sentence_transformers import SentenceTransformer
clip = SentenceTransformer("clip-ViT-B-32")        # runs on CPU

# CLIP's text encoder is capped at 77 tokens; Flipkart descriptions run to ~700,
# so we force truncation at tokenisation time (see run_clip_zeroshot.py).

prompts = [f"a photo of a product in the category: {c}" for c in cats]
label_emb = clip.encode(prompts, normalize_embeddings=True)       # (7, 512)

img_emb = clip.encode(images, normalize_embeddings=True)          # (n, 512)
txt_emb = clip.encode(descriptions, normalize_embeddings=True)    # (n, 512)

pred_img  = (img_emb @ label_emb.T).argmax(1)                     # zero-shot, image
pred_txt  = (txt_emb @ label_emb.T).argmax(1)                     # zero-shot, text
fused     = img_emb + txt_emb                                     # fusion = a mean,
fused    /= np.linalg.norm(fused, axis=1, keepdims=True)          #   same shared space
pred_fuse = (fused @ label_emb.T).argmax(1)

### Measured output (zero-shot, n=1,050, no training)
```
CLIP zero-shot · text only (truncated)   acc=0.635  macroF1=0.636
CLIP zero-shot · image only              acc=0.691  macroF1=0.673
CLIP zero-shot · image + text (trunc)    acc=0.716
CLIP zero-shot · image + text (pool)     acc=0.722   <- best zero-shot
```
![CLIP zero-shot vs supervised 2023](../docs/figures/clip_zeroshot.png)
![CLIP zero-shot confusion matrix](../docs/figures/clip_zeroshot_confusion.png)

## 2 · The honest read — and the supervised result that matters

**Zero-shot first.** CLIP reaches **~72%** on 7 classes **with no labelled data**
(image+text fusion). That stays a few points behind the *trained* 2023 pipeline,
so zero-shot CLIP does **not** beat a supervised model — what it buys is a
different cost structure: no training, no labels, fusion for free (a mean in the
shared space). For a cold-start catalogue with no labels, 72% out of the box
beats collecting and labelling a training set.

| Zero-shot approach | Training | Accuracy |
|---|:--:|:--:|
| CLIP zero-shot — image + text (pool) | **none** | **0.722** |
| Supervised 2023 — VGG16 + SVM | trained | 0.857 ± 0.022 |
| Supervised 2023 — late fusion | trained | 0.889 ± 0.021 |

**The decisive comparison is supervised-vs-supervised.** Train a linear head on
CLIP features under the *same* repeated 5-fold CV × 5 seeds used in notebook 02,
and CLIP jumps ahead of the 2023 pipeline:

| Supervised (same CV protocol) | Features | Accuracy |
|---|---|:--:|
| 2023 — late fusion | Word2Vec + VGG16 | 0.889 ± 0.021 |
| 2026 — CLIP image | CLIP image | 0.900 ± 0.016 |
| 2026 — CLIP text | CLIP text | 0.952 ± 0.017 |
| **2026 — CLIP image + text** | **CLIP fusion** | **0.953 ± 0.018** |

**At equal supervision, CLIP takes the task from 0.889 → 0.953.** Same metric,
same CV protocol, same data — only the encoder changes. That is the real 2026
argument: not "zero-shot wins", but "one pretrained multimodal encoder beats the
hand-built two-tower 2023 pipeline, and fuses for free".

## 2-bis · The only fair 2023-vs-2026 comparison (same metric: ARI)

Accuracy (here) and ARI (2023 phase 1) are different metrics on different
regimes — comparing them directly would be dishonest. So CLIP is re-scored on the
**exact 2023 unsupervised protocol**: `features → PCA → t-SNE → K-Means(k=7) →
ARI`, with `random_state=6`. Nothing else changes — only the encoder.

In [ ]:
# CLIP embeddings re-scored on the 2023 UNSUPERVISED protocol, now in PCA space
# (clustering in PCA, not t-SNE), reported as mean +/- std over 5 seeds.
# ari_text  = ari_pca_protocol(txt_emb)
# ari_image = ari_pca_protocol(img_emb)
# ari_fused = ari_pca_protocol(fused)       # text+image in one shared space
print('''   ARI (PCA) · CLIP text        = 0.47 +/- 0.02   (2023 Word2Vec PCA = 0.18 +/- 0.00)
   ARI (PCA) · CLIP image       = 0.54 +/- 0.02   (2023 VGG16    PCA = 0.49 +/- 0.01)
   ARI (PCA) · CLIP text+image  = 0.68 +/- 0.03   (no 2023 equivalent)''')

### Like-for-like result (unsupervised ARI, clustering in PCA space)
| Unsupervised regime (ARI, PCA · mean ± std) | 2023 | 2026 — CLIP |
|---|:--:|:--:|
| Text | Word2Vec 0.18 ± 0.00 | **0.47 ± 0.02** |
| Image | VGG16 0.49 ± 0.01 | **0.54 ± 0.02** |
| Text + image (shared space) | *n/a in 2023* | **0.68 ± 0.03** |
![2023 vs CLIP at equal metric](../docs/figures/ari_2023_vs_clip.png)
Same metric, same PCA pipeline, same data: CLIP lifts ARI by **+0.29 (text)** and
**+0.05 (image)**, and fuses to **0.68** with a plain mean in one shared space —
which in 2023 required the whole early-vs-late fusion study (notebook 02). (Both
sides are scored with clustering in PCA space, not t-SNE, so the numbers are
comparable and honest.)

## 3 · A real limitation — and a negative result

CLIP's text encoder is hard-capped at **77 tokens**. Flipkart descriptions run up
to ~700 tokens, so the text side is **truncated** and loses information — part of
why text-only zero-shot (0.635) is the weakest of the three.

The obvious fix is **chunk-and-pool**: split the long description into ≤77-token
chunks, embed each, and mean-pool. **It does not help here** — chunk-pooled text
lands at **0.631**, statistically on top of the truncated 0.635 (no gain), and the
pooled fusion only nudges 0.716 → 0.722. On this keyword-style catalogue the first
77 tokens already carry the category signal, so pooling the tail adds noise, not
information. An honest negative: the easy trick does not move the needle; a
genuine long-context multimodal encoder would be the real next step.

## 4 · What I keep from 2023

Not everything should change. The *method* still holds:
- the **fair, shared evaluation protocol** across approaches;
- **fixed seeds** (and `workers=1` for Word2Vec) + a held-out test set;
- **honest negatives** — the small fusion gain in notebook 02, the 77-token cap here.

Modern models change the *tools*, not the *rigour*.